In [1]:
import pandas as pd
import re
from urllib.parse import urlparse

df = pd.read_csv('../dataset/balanced_links.csv')
print(df.shape)
df.head()

(446176, 3)


,url,type,label
0,asamcgill.com/,benign,0
1,http://www.bogdanobradovic.com/t3-assets/js/20...,phishing,1
2,http://noticias.uol.com.br/tabloide/ultimas-no...,benign,0
3,manta.com/c/mt4crnc/sue-kelly-dermatology-llc,benign,0
4,ofmenandangels.blogspot.com/2009/02/vince-azzo...,benign,0


In [2]:
# Normalize all URLs - strip protocol prefix to ensure consistency
def normalize_url(url):
    url = str(url).strip()
    url = url.replace('https://', '').replace('http://', '')
    return url

df['url_normalized'] = df['url'].apply(normalize_url)

print("Sample normalized URLs:")
print(df[['url', 'url_normalized']].head(10).to_string())
print("\nOriginal URL sample with http://:")
print(df[df['url'].str.startswith('http')][['url', 'url_normalized']].head(5).to_string())

Sample normalized URLs:
                                                                                                                                                         url                                                                                                                                      url_normalized
0                                                                                                                                             asamcgill.com/                                                                                                                                      asamcgill.com/
1                                                                                            http://www.bogdanobradovic.com/t3-assets/js/2014/abt/acesso.php                                                                                            www.bogdanobradovic.com/t3-assets/js/2014/abt/acesso.php
2  http://noticias.uol.com.br/tabloide/ultimas-noticias/tablo

In [3]:
def extract_url_features(url):
    features = {}
    
    # Ensure url is a string
    url = str(url)
    
    # Parse the URL
    try:
        parsed = urlparse(url if url.startswith('http') else 'http://' + url)
        domain = parsed.netloc
        path = parsed.path
    except:
        domain = ''
        path = ''
    
    # Length features
    features['url_length'] = len(url)
    features['domain_length'] = len(domain)
    features['path_length'] = len(path)
    
    # Count-based features
    features['num_dots'] = url.count('.')
    features['num_hyphens'] = url.count('-')
    features['num_slashes'] = url.count('/')
    features['num_at'] = url.count('@')
    features['num_digits'] = sum(c.isdigit() for c in url)
    features['num_params'] = url.count('?')
    features['num_percent'] = url.count('%')
    features['num_subdomains'] = len(domain.split('.')) - 2 if domain else 0
    
    # Binary features
    features['has_ip'] = 1 if re.match(r'\d+\.\d+\.\d+\.\d+', domain) else 0
    features['has_https'] = 1 if url.startswith('https') else 0
    features['has_at_symbol'] = 1 if '@' in url else 0
    features['has_double_slash'] = 1 if '//' in url[7:] else 0
    
    # Suspicious keyword features
    suspicious_keywords = ['login', 'verify', 'secure', 'account', 'update',
                           'banking', 'confirm', 'password', 'signin', 'ebayisapi',
                           'webscr', 'cmd', 'dispatch', 'paypal']
    for keyword in suspicious_keywords:
        features[f'has_{keyword}'] = 1 if keyword in url.lower() else 0
    
    return features

In [4]:
print("Extracting URL features...")
url_features = df['url_normalized'].apply(extract_url_features)
url_features_df = pd.DataFrame(url_features.tolist())

print("Done! Shape:", url_features_df.shape)
url_features_df.head()

Extracting URL features...
Done! Shape: (446176, 29)


,url_length,domain_length,path_length,num_dots,num_hyphens,num_slashes,num_at,num_digits,num_params,num_percent,...,has_update,has_banking,has_confirm,has_password,has_signin,has_ebayisapi,has_webscr,has_cmd,has_dispatch,has_paypal
0,14,13,1,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,56,23,33,3,1,5,0,5,0,0,...,0,0,0,0,0,0,0,0,0,0
2,146,19,127,4,13,7,0,8,0,0,...,0,0,0,0,0,0,0,0,0,0
3,45,9,36,1,3,3,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,56,27,29,3,1,3,0,6,0,0,...,0,0,0,0,0,0,0,0,0,0


In [5]:
# Combine features with labels
df_url_features = pd.concat([url_features_df, df['label'].reset_index(drop=True)], axis=1)

print("Final shape:", df_url_features.shape)
print("\nMissing values:\n", df_url_features.isnull().sum().sum())
df_url_features.head()

Final shape: (446176, 30)

Missing values:
 0


,url_length,domain_length,path_length,num_dots,num_hyphens,num_slashes,num_at,num_digits,num_params,num_percent,...,has_banking,has_confirm,has_password,has_signin,has_ebayisapi,has_webscr,has_cmd,has_dispatch,has_paypal,label
0,14,13,1,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,56,23,33,3,1,5,0,5,0,0,...,0,0,0,0,0,0,0,0,0,1
2,146,19,127,4,13,7,0,8,0,0,...,0,0,0,0,0,0,0,0,0,0
3,45,9,36,1,3,3,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,56,27,29,3,1,3,0,6,0,0,...,0,0,0,0,0,0,0,0,0,0


In [6]:
df_url_features.to_csv('../dataset/url_features.csv', index=False)
print("Saved successfully!")

Saved successfully!
